# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratham6306/ml-internship-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# ==========================
# ML-04 : Data Contract
# Imports
# ==========================

!pip -q install duckdb huggingface_hub pandas pyarrow

import duckdb
import pandas as pd

from huggingface_hub import login
from google.colab import userdata

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

### Data Contract

**Unit of analysis**

One row represents the daily performance of one content item for one client on one report date.

**Table used**

`fact_content_daily_performance`

**Time window**

March 2026 (2026-03-01 to 2026-03-31)

**Prediction / ranking objective**

Predict future content performance using historical search and analytics metrics.

**Deliberately excluded**

Any future information or label-derived columns are excluded because they would introduce data leakage.

In [15]:
grain_check = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_path}')
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate groups found:", len(grain_check))
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate groups found: 0


,client_hash_id,content_hash_id,report_date,duplicate_rows


In [16]:
summary = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
""").df()

summary

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [17]:
availability = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows
FROM read_parquet('{march_path}')
""").df()

availability

,total_rows,gsc_rows,ga4_rows
0,9841378,3611061.0,413966.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
import os
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

# Make the token available to DuckDB
os.environ["HF_TOKEN"] = HF_TOKEN

print("✅ Hugging Face login successful")

✅ Hugging Face login successful


In [18]:
import pandas as pd

field_contract = pd.DataFrame({
    "Field": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "ga4_users",
        "ga4_engaged_sessions",
        "ga4_total_engagement_sec",
        "gsc_data_available",
        "ga4_data_available",
        "month"
    ],
    "Category": [
        "Context",
        "Context",
        "Context",
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Feature",
        "Context",
        "Context",
        "Context"
    ],
    "Reason": [
        "Client identifier; never used as model input.",
        "Content identifier; never used as model input.",
        "Observation date.",
        "Historical search metric available before prediction.",
        "Historical search metric available before prediction.",
        "Historical ranking metric available before prediction.",
        "Historical GA4 metric available before prediction.",
        "Historical GA4 metric available before prediction.",
        "Historical GA4 metric available before prediction.",
        "Historical GA4 engagement metric.",
        "Historical engagement duration.",
        "Indicates whether GSC metrics are valid.",
        "Indicates whether GA4 metrics are valid.",
        "Partition column for filtering."
    ]
})

field_contract

,Field,Category,Reason
0,client_hash_id,Context,Client identifier; never used as model input.
1,content_hash_id,Context,Content identifier; never used as model input.
2,report_date,Context,Observation date.
3,gsc_impressions,Feature,Historical search metric available before prediction.
4,gsc_clicks,Feature,Historical search metric available before prediction.
5,gsc_avg_position,Feature,Historical ranking metric available before prediction.
6,ga4_pageviews,Feature,Historical GA4 metric available before prediction.
7,ga4_sessions,Feature,Historical GA4 metric available before prediction.
8,ga4_users,Feature,Historical GA4 metric available before prediction.
9,ga4_engaged_sessions,Feature,Historical GA4 engagement metric.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Create DuckDB connection

con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.3.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
from huggingface_hub import hf_hub_download

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [13]:
df = con.execute(f"""
SELECT *
FROM read_parquet('{march_path}')
""").df()

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,3.350000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,0.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,4.928000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,4.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,2.272727,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [14]:
print(df.columns.tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [19]:
limitations = [
    "This notebook uses only the March 2026 partition, so seasonal patterns across the full panel are not evaluated.",
    "Clients have different history lengths, which may affect comparisons.",
    "Rows where gsc_data_available or ga4_data_available is FALSE should not be interpreted as genuine zero activity.",
    "This analysis is intended for decision support and cannot establish causal relationships."
]

for i, item in enumerate(limitations, start=1):
    print(f"{i}. {item}")

1. This notebook uses only the March 2026 partition, so seasonal patterns across the full panel are not evaluated.
2. Clients have different history lengths, which may affect comparisons.
3. Rows where gsc_data_available or ga4_data_available is FALSE should not be interpreted as genuine zero activity.
4. This analysis is intended for decision support and cannot establish causal relationships.


In [20]:
missing = df.isnull().mean().sort_values(ascending=False) * 100

missing = missing.reset_index()
missing.columns = ["Column", "Missing (%)"]

display(missing)

,Column,Missing (%)
0,gsc_avg_position,63.307364
1,ga4_users,30.673967
2,ga4_data_available,30.673967
3,ga4_pageviews,30.673967
4,ga4_sessions,30.673967
5,ga4_engaged_sessions,30.673967
6,ai_perplexity,30.673967
7,ai_chatgpt,30.673967
8,sessions_ai,30.673967
9,sessions_paid,30.673967


### Limitations

- Only the March 2026 partition is analyzed; seasonal effects across the full panel are not considered.
- Clients have different history lengths, which may influence comparisons.
- Rows with `gsc_data_available = FALSE` or `ga4_data_available = FALSE` should not be interpreted as genuine zero activity.
- This analysis supports decision making but does not establish causal relationships.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.